# 02 — LLaMEA Synthesis

**GPT-6 Luna token pilot · f10 · 10D · Explicit Baseline · heteroscedastic noise σ = 0.2**

Saved outputs may reflect earlier runs. Use separate study storage for an independent experiment.


In [5]:
import os
import sys
from pathlib import Path

# Ensure project root & src are in path
cwd = Path(".").resolve()
root_dir = cwd.parent if cwd.name == "notebooks" else cwd
os.chdir(root_dir)
src_dir = root_dir / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

# Keep module identities stable for multiprocessing/pickling.
# After updating project code or configuration, restart the kernel instead of evicting modules.

from evolution.application.model_selection import ModelSelection
from evolution.infra.di import EvolutionProvider
from evolution.application.campaign.audit import CampaignAudit
from evolution.application.campaign.service import SynthesisCampaign
from shared.infra.lifecycle import WorkflowScope

print("[OK] LLaMEA Evolution Pipeline initialized.")

# Select the provider and exact model for this campaign.
model = ModelSelection(provider="upb", model_id="openai.gpt-6-luna")

# To use a local server, edit both fields, for example:
# model = ModelSelection(provider="local", model_id="qwen2.5-coder-32b-instruct-q4_k_m.gguf")


[OK] LLaMEA Evolution Pipeline initialized.


## 1. Connect the model


In [6]:
# Prepare services once; infrastructure owns shutdown cleanup.
if "services" in globals():
    services.close()
services = WorkflowScope(EvolutionProvider(), context={}, request_context={})
campaign = services.get(SynthesisCampaign)
audit = services.get(CampaignAudit)
print("=" * 65)
print(f"Provider: {model.provider}")
print(f"Model ID: {model.model_id}")
print("=" * 65)

Diagnostic log: /Users/nicolaibrahim/Desktop/proj/AAD_LLM/logs/evolution/2026-11-21_09-47-01_UTC-662dcbec/diagnostics.log
Provider: upb
Model ID: openai.gpt-6-luna


## 2. Audit synthesis coverage

Check the selected matrix against the current database before running.


In [7]:
matrix_df_result = audit.inspect(
    model=model,
    problem_targets=[
           {"id": p, "dimensions": [10]}
           for p in [1, 8, 11, 15, 21]
       ],
       noise_conditions=[
           {"std": std, "noise_model": "heteroscedastic"}
           for std in [0.0, 0.2]
       ],
       synthesis_modes=[
        {"mode": "explicit", "strategies": ["baseline", "vectorization"]},
        {"mode": "implicit", "strategies": ["baseline", "vectorization"]},
    ],
    iterations=10,
    budget=100_000,
    timeout_seconds=30.0,
    runs_per_config=1,
    auto_resume=True,
    skip_completed=True,
    retry_failed_synthesis=True,
)
if matrix_df_result.is_success:
    matrix_df, summary = matrix_df_result.value
    display(matrix_df) if "display" in globals() else print(matrix_df.to_string())
else:
    matrix_df = None
    summary = None
    print(f"Error: {matrix_df_result.error.message}")



🎯 Synthesis Matrix Audit for 'openai.gpt-6-luna':
                                                        Target Runs  Completed  Running Records  Failed Records      Status
Problem         Dimension Environment    Strategy                                                                          
f1 (Sphere)     10D       Explicit (0.0) Baseline                 1          0                0               0   ⏳ Pending
                                         Vectorization            1          0                0               0   ⏳ Pending
                          Implicit (0.0) Baseline                 1          0                0               0   ⏳ Pending
                                         Vectorization            1          0                0               0   ⏳ Pending
                          Explicit (0.2) Baseline                 1          1                0               0  ✅ Complete
                                         Vectorization            1          1   

## 3. Run synthesis

Completed conditions are skipped; interrupted runs require their checkpoints to resume.


In [8]:
# Run one ten-generation experiment and inspect the Gateway token usage lines.
results_result = campaign.run(
    model=model,
    problem_targets=[
        {"id": p, "dimensions": [50]}
        for p in [1, 8, 11, 15, 21]
    ],
    noise_conditions=[
        {"std": std, "noise_model": "heteroscedastic"}
        for std in [0.0, 0.2]
    ],
    synthesis_modes=[
        {"mode": "explicit", "strategies": ["baseline", "vectorization"]},
        {"mode": "implicit", "strategies": ["baseline", "vectorization"]},
    ],
    iterations=10,
    budget=100_000,
    timeout_seconds=30.0,
    runs_per_config=1,
    auto_resume=True,
    skip_completed=True,
    retry_failed_synthesis=True,
)
results = results_result.on(
    on_success=lambda value: value,
    on_failure=lambda error: print(f"Error: {error.message}"),
)



🧬 LLAMEA EVOLUTIONARY ALGORITHM SYNTHESIS
   Model: openai.gpt-6-luna | Pending Tasks: 29 | Concurrency: 8 workers

[1/1] 🧬 Task: openai.gpt-6-luna | Dim: 50D | Noise: clean (σ=0.0) | Problem: f8 (Rosenbrock) | Strategy: Vectorization | Exp ID: #7403

[1/1] 🧬 Task: openai.gpt-6-luna | Dim: 50D | Noise: clean (σ=0.0) | Problem: f1 (Sphere) | Strategy: Baseline | Exp ID: #7398

[1/1] 🧬 Task: openai.gpt-6-luna | Dim: 50D | Noise: clean (σ=0.0) | Problem: f8 (Rosenbrock) | Strategy: Baseline | Exp ID: #7404
Gateway token usage for openai.gpt-6-luna: input=1387 cached_input=0 output=3868 reasoning=2153 total=5255 finish=stop
  • ⚡ Gen  1/10 | Algo: AlgorithmName      | Error:   2.5280e+01 | Fitness:  -2.5280e+01 | Evals: 100,000 | Time:  1.64s

[1/1] 🧬 Task: openai.gpt-6-luna | Dim: 50D | Noise: clean (σ=0.0) | Problem: f1 (Sphere) | Strategy: Vectorization | Exp ID: #7399
Gateway token usage for openai.gpt-6-luna: input=1429 cached_input=1426 output=3910 reasoning=2245 total=5339 finish=s

   Failed / Incomplete       : 0

